# 04 CNN Model

This notebook builds and evaluates a simple 1D CNN for software effort prediction.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.evaluate import evaluate_predictions, save_metrics

root_dir = Path.cwd()
processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

candidate_files = [
    processed_dir / "desharnais_processed.csv",
    processed_dir / "cocomo81_processed.csv",
    processed_dir / "china_processed.csv",
]

dataset_path = next((p for p in candidate_files if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Run 02_preprocessing.ipynb first to create processed datasets")

df = pd.read_csv(dataset_path)
target_col = df.columns[-1]
X = df.drop(columns=[target_col]).values.astype(np.float32)
y = df[target_col].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_cnn = reshape_for_cnn(X_train)
X_test_cnn = reshape_for_cnn(X_test)

print("Using dataset:", dataset_path.name)
print("CNN input shape:", X_train_cnn.shape)

In [ ]:
model = build_cnn_regressor(
    input_length=X_train_cnn.shape[1],
    filters=32,
    kernel_size=3,
    dense_units=64,
    learning_rate=1e-3,
 )

model, history = train_cnn_model(
    model,
    X_train_cnn, y_train,
    X_test_cnn, y_test,
    epochs=50,
    batch_size=32,
    verbose=0,
 )

print("Final validation MAE:", history["val_mean_absolute_error"][-1])

In [ ]:
cnn_preds = model.predict(X_test_cnn).ravel()
cnn_metrics = evaluate_predictions("cnn", y_test, cnn_preds)
cnn_metrics

In [ ]:
cnn_metrics_path = save_metrics(
    cnn_metrics,
    file_name="cnn_metrics.csv",
    output_dir=metrics_dir,
 )
print("Saved CNN metrics to:", cnn_metrics_path)